In [ ]:
# Install required package (run this only once)
!pip install scikit-learn matplotlib pandas


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load dataset
try:
    data = pd.read_csv("agents_physiological_data.csv")
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: agents_physiological_data.csv not found.")
    raise


In [ ]:
# Create 'stress' label based on physiological data
data['stress'] = ((data['heart_rate'] > 100) | (data['systolic_bp'] > 130))
data['stress'] = data['stress'].astype(int)  # Convert to 0 or 1


In [ ]:
# Feature selection
features = data[['heart_rate', 'hrv', 'systolic_bp', 'diastolic_bp', 'body_temp']]
labels = data['stress']

# Normalize features
scaler = MinMaxScaler()
features_normalized = scaler.fit_transform(features)


In [ ]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    features_normalized, labels, test_size=0.2, random_state=42)


In [ ]:
# Train RandomForest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
# Plot true vs predicted stress
plt.figure(figsize=(8, 5))
plt.scatter(range(len(y_test)), y_test, color='blue', label='True Stress')
plt.scatter(range(len(y_pred)), y_pred, color='pink', alpha=0.6, label='Predicted Stress')
plt.xlabel('Sample Index')
plt.ylabel('Stress (True: 1, False: 0)')
plt.title('Predicted Stress vs True Stress')
plt.legend()
plt.show()


In [ ]:
# Function to predict stress from user input
def predict_stress():
    try:
        print("Enter the following data for the agent:")

        heart_rate = float(input("Heart Rate (bpm): "))
        hrv = float(input("HRV (ms): "))
        systolic_bp = float(input("Systolic BP (mmHg): "))
        diastolic_bp = float(input("Diastolic BP (mmHg): "))
        body_temp = float(input("Body Temperature (°C): "))

        # Validate inputs
        if not (30 < heart_rate < 200):
            raise ValueError("Unrealistic heart rate.")
        if not (0 < hrv < 300):
            raise ValueError("Unrealistic HRV.")
        if not (50 < systolic_bp < 250):
            raise ValueError("Unrealistic systolic BP.")
        if not (30 < diastolic_bp < 150):
            raise ValueError("Unrealistic diastolic BP.")
        if not (30 < body_temp < 45):
            raise ValueError("Unrealistic body temperature.")

        new_data = [[heart_rate, hrv, systolic_bp, diastolic_bp, body_temp]]
        new_data_normalized = scaler.transform(new_data)
        stress_prediction = clf.predict(new_data_normalized)

        if stress_prediction[0] == 1:
            print("\nThe agent is stressed.\nYour delivery is being given to other available agents.")
        else:
            print("\nThe agent is not stressed.\nYour order will reach you soon.")

    except ValueError as ve:
        print(f"Input error: {ve}")
    except Exception as e:
        print(f"Something went wrong: {e}")


In [ ]:
# Run prediction
predict_stress()
